# SteelFlow: demand and sales forecasting EDA

The demand grain is one SKU at one demand centre on one target date. We validate coverage, seasonality, sparsity, and the supplied horizon feature marts.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
con = duckdb.connect()
sns.set_theme(style="whitegrid")


In [ ]:
demand = con.execute(f"SELECT * FROM read_parquet('{(DATA/'daily_demand_by_sku_centre.parquet').as_posix()}')").fetchdf()
demand['target_date'] = pd.to_datetime(demand['target_date'])
demand.info()

In [ ]:
demand[['demand_kg','gross_requested_quantity_kg','accepted_order_quantity_kg']].describe(percentiles=[.01,.5,.95,.99])

In [ ]:
daily = demand.groupby('target_date', as_index=False)['demand_kg'].sum()
plt.figure(figsize=(14, 4)); sns.lineplot(data=daily, x='target_date', y='demand_kg'); plt.title('Total daily demand'); plt.show()

In [ ]:
pivot = demand.pivot_table(index='target_date', columns='product_family', values='demand_kg', aggfunc='sum')
plt.figure(figsize=(14, 6)); pivot.rolling(30).mean().plot(ax=plt.gca()); plt.title('30-day demand by product family'); plt.show()

In [ ]:
h07 = con.execute(f"SELECT * FROM read_parquet('{(DATA/'forecast_features_horizon_07.parquet').as_posix()}')").fetchdf()
for col in ['origin_date','feature_cutoff_date','target_date']:
    h07[col] = pd.to_datetime(h07[col])
print(h07.groupby('split').agg(rows=('forecast_row_id','size'), origin_min=('origin_date','min'), origin_max=('origin_date','max'), target_mean=('target_demand_kg','mean')))

In [ ]:
# Leakage checks: target must be after cutoff, and feature availability flags should explain missing lags
assert (h07['target_date'] > h07['feature_cutoff_date']).all()
assert (h07['target_date'] >= h07['origin_date']).all()
availability = h07.filter(regex='available$').mean().sort_values()
availability

## Modeling handoff

The supplied horizon tables already contain train/validation/test assignments and prediction-time features, so no integrated dataset is required for the first forecasting benchmark. The model notebook will use these tables directly and compare a seasonal-naive benchmark with tree ensembles.